# Классификация цвета автомобилей

Сравнить предобученные ResNet50 и ConvNeXt-Tiny с небольшой CNN, обучаемой с нуля, на несбалансированной классификации цветов.

[Описание и данные](README.md) · [История эксперимента](results/recorded_metrics.json)

**Запуск:** Python 3.11 или 3.12. Полное обучение требует данных и подходящего GPU. Выводы очищены; численные результаты ниже появятся после выполнения.

## Подготовка окружения

Ячейка устанавливает зависимости проекта. Локально рекомендуется отдельное виртуальное окружение.

In [ ]:
%pip -q install -r https://raw.githubusercontent.com/Xokage9/ml-portfolio/main/projects/05_transfer_learning/requirements.txt


## Окружение и воспроизводимость

In [ ]:
import os, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights


plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"MPS  available:  {torch.backends.mps.is_available()}")


## Подготовка и аудит набора данных

In [ ]:
IMAGE_TABLE_PATH = Path('data/car_colors/1/Image_table.csv')
IMAGES_DIR = Path('data/car_colors/1/confirmed_fronts')

image_df = pd.read_csv(str(IMAGE_TABLE_PATH))
image_df.columns = [c.strip() for c in image_df.columns]
print(f'Исходное число строк: {len(image_df):,}')

def parse_image_name(name):
    parts = str(name).split('$$')
    if len(parts) >= 4:
        return {
            'color': parts[3].strip(),
            'quality': parts[4].strip() if len(parts) > 4 else '',
            'angle': parts[5].strip() if len(parts) > 5 else ''
        }
    return {'color': '', 'quality': '', 'angle': ''}

parsed = image_df['Image_name'].apply(parse_image_name)
image_df['parsed_color'] = parsed.apply(lambda x: x['color'])
image_df['parsed_quality'] = parsed.apply(lambda x: x['quality'])
image_df['parsed_angle'] = parsed.apply(lambda x: x['angle'])


USE_QUALITY_ONLY = True
if USE_QUALITY_ONLY and 'Quality_check' in image_df.columns:
    image_df = image_df[image_df['Quality_check'].fillna('').astype(str).str.strip().ne('N')].copy()
print(f'После фильтра quality: {len(image_df):,}')


all_image_paths = list(IMAGES_DIR.rglob('*.jpg'))
image_name_to_path = {path.name: path for path in all_image_paths}
print(f'Найдено jpg файлов: {len(all_image_paths):,}')

image_df['image_path'] = image_df['Image_name'].map(image_name_to_path)
image_df = image_df.dropna(subset=['image_path']).copy()
print(f'После mapping: {len(image_df):,}')


color_to_idx = {color: idx for idx, color in enumerate(sorted(image_df['parsed_color'].unique()))}
idx_to_color = {idx: color for color, idx in color_to_idx.items()}
image_df['label'] = image_df['parsed_color'].map(color_to_idx)
num_classes_initial = len(color_to_idx)
print(f'Число классов до очистки: {num_classes_initial}')


class_counts = image_df['label'].value_counts()
MIN_SAMPLES_PER_CLASS = 5
valid_labels = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index

if len(valid_labels) < len(class_counts):
    removed_count = len(class_counts) - len(valid_labels)
    print(f'[WARNING] Удалено {removed_count} классов с количеством образцов < {MIN_SAMPLES_PER_CLASS}')

image_df = image_df[image_df['label'].isin(valid_labels)].reset_index(drop=True)


unique_colors = sorted(image_df['parsed_color'].unique())
color_to_idx = {color: idx for idx, color in enumerate(unique_colors)}
idx_to_color = {idx: color for color, idx in color_to_idx.items()}
image_df['label'] = image_df['parsed_color'].map(color_to_idx)
num_classes = len(color_to_idx)

print(f'\n{"="*50}')
print(f'Итоговое число классов: {num_classes}')
print(f'Итоговое число строк: {len(image_df):,}')
print(f'{"="*50}')


plt.figure(figsize=(14, 6))
class_distribution = image_df['parsed_color'].value_counts()
plt.bar(range(len(class_distribution)), class_distribution.values)
plt.xticks(range(len(class_distribution)), class_distribution.index, rotation=90, fontsize=8)
plt.xlabel('Цвет автомобиля')
plt.ylabel('Количество изображений')
plt.title('Распределение классов в датасете')
plt.tight_layout()
plt.show()


## Разбиение и аугментации

In [ ]:

train_df, val_df = train_test_split(
    image_df,
    test_size=0.2,
    random_state=42,
    stratify=image_df['label']
)

print(f'Train size: {len(train_df):,} ({len(train_df)/len(image_df)*100:.1f}%)')
print(f'Val size: {len(val_df):,} ({len(val_df)/len(image_df)*100:.1f}%)')


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CarColorDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label = row['label']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = CarColorDataset(train_df, transform=train_transforms)
val_dataset = CarColorDataset(val_df, transform=val_transforms)

_pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=_pin)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=_pin)

print(f"\nBatch size: 32")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


## Цикл обучения и метрики

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1, all_preds, all_labels

def train_model(model, model_name, train_loader, val_loader, num_classes, epochs=10, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_f1 = -1.0
    best_model_state = None

    print(f"\n{'='*60}")
    print(f"Обучение модели: {model_name}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_f1, _, _ = validate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"\nЛучший F1 Macro: {best_f1:.4f}")
    print(f"{'='*60}\n")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history, best_f1

def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{model_name} - Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['train_f1'], label='Train F1', marker='o')
    axes[1].plot(history['val_f1'], label='Val F1', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Macro')
    axes[1].set_title(f'{model_name} - F1 Macro')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(all_labels, all_preds, model_name, class_names=None):
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.tight_layout()
    plt.show()


## Fine-tuning ResNet50

In [ ]:
print("\n" + "="*60)
print("МОДЕЛЬ 1: ResNet50 (Pretrained на ImageNet)")
print("="*60)

model1 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)


for param in model1.parameters():
    param.requires_grad = False


num_features = model1.fc.in_features
model1.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, num_classes)
)


for param in model1.layer4.parameters():
    param.requires_grad = True

for param in model1.fc.parameters():
    param.requires_grad = True


model1 = model1.to(device)


trainable_params = sum(p.numel() for p in model1.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model1.parameters())
print(f"Обучаемых параметров: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.1f}%)")


if trainable_params < 1_000_000:
    for param in model1.layer3.parameters():
        param.requires_grad = True


criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model1.parameters()),  # Только обучаемые параметры
    lr=0.001,
    weight_decay=0.01
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)


BATCH_SIZE = 32
_pin = torch.cuda.is_available()
train_loader_finetune = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=_pin)
val_loader_finetune = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=_pin)

print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")


def train_simple(model, train_ld, val_ld, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_f1 = -1.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        all_preds, all_labels = [], []

        for batch_idx, (imgs, lbls) in enumerate(train_ld):
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

            if (batch_idx + 1) % 50 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_ld)} | Loss: {loss.item():.4f}")

        train_acc = correct / total
        train_f1 = f1_score(all_labels, all_preds, average='macro')
        avg_train_loss = train_loss / len(train_ld)
        scheduler.step()

        model.eval()
        val_loss, correct, total = 0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, lbls in val_ld:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbls)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                correct += (preds == lbls).sum().item()
                total += lbls.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())

        val_acc = correct / total
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        avg_val_loss = val_loss / len(val_ld)


        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)


        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"  Train → Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"  Val   → Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ★ Новый лучший результат: {best_f1:.4f}")

    if best_model_state is not None:
        model.load_state_dict({k: v.to(next(model.parameters()).device) for k, v in best_model_state.items()})

    return model, history, best_f1

print("\n[Начало обучения...]")
model1, history1, best_f1_1 = train_simple(
    model1,
    train_loader_finetune,
    val_loader_finetune,
    epochs=5,
    lr=0.001
)

print(f"\nОбучение завершено! Лучший F1 Macro: {best_f1_1:.4f}")


val_loss1, val_f1_1, preds1, labels1 = validate(model1, val_loader_finetune, nn.CrossEntropyLoss().to(device), device)
print(f"Финальный F1 Macro (ResNet50): {val_f1_1:.4f}")
plot_history(history1, "ResNet50 (ImageNet)")


## Fine-tuning ConvNeXt-Tiny

In [ ]:
print("\n" + "="*60)
print("МОДЕЛЬ 2: ConvNeXt-Tiny (Pretrained на ImageNet-22K)")
print("="*60)
model2 = timm.create_model('convnext_tiny.fb_in22k', pretrained=True, num_classes=num_classes)

trainable2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
total2 = sum(p.numel() for p in model2.parameters())
print(f"Параметров: {trainable2:,} / {total2:,} ({trainable2/total2*100:.1f}% обучаемых)")

model2, history2, best_f1_2 = train_model(
    model2,
    "ConvNeXt-Tiny (ImageNet-22K)",
    train_loader,
    val_loader,
    num_classes,
    epochs=10,
    lr=0.0005
)

val_loss2, val_f1_2, preds2, labels2 = validate(model2, val_loader, nn.CrossEntropyLoss(), device)
print(f"Финальный F1 Macro (ConvNeXt-Tiny): {val_f1_2:.4f}")

plot_history(history2, "ConvNeXt-Tiny (ImageNet-22K)")


## Обучение CNN с нуля

In [ ]:
import torch
import torch.nn as nn

print("\n" + "="*60)
print("МОДЕЛЬ 3: ScratchCNN-Lite (с нуля, быстрое обучение)")
print("="*60)

class DSConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, p_drop=0.1):
        super().__init__()
        self.block = nn.Sequential(
            # depthwise
            nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.SiLU(inplace=True),

            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),

            nn.Dropout2d(p_drop)
        )

    def forward(self, x):
        return self.block(x)


class ScratchCNNLite(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),  # 224 -> 112
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True)
        )

        self.features = nn.Sequential(
            DSConvBlock(32, 64, stride=1, p_drop=0.05),   # 112
            DSConvBlock(64, 128, stride=2, p_drop=0.10),  # 56
            DSConvBlock(128, 128, stride=1, p_drop=0.10), # 56
            DSConvBlock(128, 256, stride=2, p_drop=0.15), # 28
            DSConvBlock(256, 256, stride=1, p_drop=0.15), # 28
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.classifier(x)
        return x


model3 = ScratchCNNLite(num_classes)

total_params = sum(p.numel() for p in model3.parameters())
trainable_params = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

model3, history3, best_f1_3 = train_model(
    model3,
    "ScratchCNN-Lite",
    train_loader,
    val_loader,
    num_classes,
    epochs=12,
    lr=0.003
)

val_loss3, val_f1_3, preds3, labels3 = validate(model3, val_loader, nn.CrossEntropyLoss(), device)
print(f"Финальный F1 Macro (ScratchCNN-Lite): {val_f1_3:.4f}")

plot_history(history3, "ScratchCNN-Lite")


## Сравнение по результатам текущего запуска

In [ ]:

results = pd.DataFrame({
    'model': ['ResNet50', 'ConvNeXt-Tiny', 'ScratchCNN-Lite'],
    'validation_macro_f1': [float(val_f1_1), float(val_f1_2), float(val_f1_3)],
    'best_epoch_macro_f1': [float(best_f1_1), float(best_f1_2), float(best_f1_3)],
    'epochs': [len(history1['val_f1']), len(history2['val_f1']), len(history3['val_f1'])],
})
results.to_csv('current_results.csv', index=False)
display(results)
for history, name in [(history1, 'ResNet50'), (history2, 'ConvNeXt-Tiny'), (history3, 'ScratchCNN-Lite')]:
    plt.plot(range(1, len(history['val_f1']) + 1), history['val_f1'], label=name)
plt.xlabel('Epoch')
plt.ylabel('Validation Macro-F1')
plt.legend()
plt.show()
print('Лучшая модель:', results.loc[results.validation_macro_f1.idxmax(), 'model'])


## Как читать результат

Сохранённый эксперимент: 61 825 изображений, 21 класс, разбиение train/validation 80/20. Таблицы новой версии строятся из `validate`, а кривые — из истории обучения. Копия лучшего state_dict теперь независима от следующих шагов обучения.

Независимая test-выборка в исходной работе отсутствует. После исправления checkpoint результаты нужно получить заново; старые числа сохранены отдельно как история эксперимента.

Сохранённая история исходного эксперимента находится отдельно в `results/recorded_metrics.json`. Полное GPU-обучение при подготовке этой версии не выполнялось.